In [14]:
import pandas as pd

#df = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
#df = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
#df = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')
df = pd.read_csv("../dataset/raw/play_off_box_scores_2010_2024.csv")

df = df.drop(columns = ['jerseyNum', 'comment'], axis = 1)
df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())
df = df.dropna()
df.info()

C:\Users\sidne\AppData\Local\Temp\ipykernel_5792\3047109016.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())


<class 'pandas.core.frame.DataFrame'>
Index: 20242 entries, 45 to 31184
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   season_year              20242 non-null  object 
 1   game_date                20242 non-null  object 
 2   gameId                   20242 non-null  int64  
 3   teamId                   20242 non-null  int64  
 4   teamCity                 20242 non-null  object 
 5   teamName                 20242 non-null  object 
 6   teamTricode              20242 non-null  object 
 7   teamSlug                 20242 non-null  object 
 8   personId                 20242 non-null  int64  
 9   personName               20242 non-null  object 
 10  position                 20242 non-null  object 
 11  minutes                  20242 non-null  object 
 12  fieldGoalsMade           20242 non-null  int64  
 13  fieldGoalsAttempted      20242 non-null  int64  
 14  fieldGoalsPercentage     2

In [16]:
#player_stats = df.drop(columns = ['minutes', 'season_year', 'game_date', 'gameId', 'matchup', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId'], axis = 1).copy()
player_stats = df.drop(columns = ['minutes', 'season_year', 'game_date', 'gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId'], axis = 1).copy()
player_positions = player_stats.groupby(by = 'personName').agg({'position': lambda x: ', '.join(x.unique())}).reset_index()
player_stats_group = player_stats.groupby(by = 'personName').sum().reset_index()
player_stats_group = player_stats_group.drop(columns = 'position', axis = 1)
#player_stats_group.sort_values(by = 'points', ascending = False)
player_stats = pd.merge(player_positions, player_stats_group, on = 'personName', how = 'outer')
player_stats['fieldGoalsPercentage'] = player_stats.apply(lambda x: (x['fieldGoalsMade'] / x['fieldGoalsAttempted']) * 100 if x['fieldGoalsAttempted'] != 0 else 0, axis = 1)
player_stats['threePointersPercentage'] = player_stats.apply(lambda x: (x['threePointersMade'] / x['threePointersAttempted']) * 100 if x['threePointersAttempted'] != 0 else 0, axis = 1)
player_stats['freeThrowsPercentage'] = player_stats.apply(lambda x: (x['freeThrowsMade'] / x['freeThrowsAttempted']) * 100 if x['freeThrowsAttempted'] != 0 else 0, axis = 1)

#player_stats.to_csv('../dataset/clean/stats_by_player_part_1.csv', index = False)
#player_stats.to_csv('../dataset/clean/stats_by_player_part_2.csv', index = False)
#player_stats.to_csv('../dataset/clean/stats_by_player_part_3.csv', index = False)
player_stats.to_csv('../dataset/clean/stats_by_player_playoffs.csv', index = False)

In [21]:
df_part1 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_1.csv')
df_part2 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_2.csv')
df_part3 = pd.read_csv('../dataset/raw/regular_season_box_scores_2010_2024_part_3.csv')
df_final = pd.concat([df_part1, df_part2, df_part3], axis = 0)
df_final = player_stats.groupby(by = 'personName').sum().reset_index()
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 485 entries, 0 to 484
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   personName               485 non-null    object 
 1   position                 485 non-null    object 
 2   fieldGoalsMade           485 non-null    int64  
 3   fieldGoalsAttempted      485 non-null    int64  
 4   fieldGoalsPercentage     485 non-null    float64
 5   threePointersMade        485 non-null    int64  
 6   threePointersAttempted   485 non-null    int64  
 7   threePointersPercentage  485 non-null    float64
 8   freeThrowsMade           485 non-null    int64  
 9   freeThrowsAttempted      485 non-null    int64  
 10  freeThrowsPercentage     485 non-null    float64
 11  reboundsOffensive        485 non-null    int64  
 12  reboundsDefensive        485 non-null    int64  
 13  reboundsTotal            485 non-null    int64  
 14  assists                  4